# M7 · Backfill `fact_maintenance` (header) + `fact_maintenance_line`

**Goal:** load vehicle-grain maintenance work-orders and their line items (offense / sinistre / reparation) into the warehouse for the BI maintenance dashboard.

**SQL files:**
- `sql/18_fact_maintenance_incr.sql` (header — UPSERT on (tenant_id, id_maintenance))
- `sql/19_fact_maintenance_line_incr.sql` (lines — DELETE-INSERT-on-window)

**Run order matters:** header first (lines reference parent `id_maintenance`).

In [1]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.config import settings, load_pipeline_config
from accent_fleet.db import get_engine
from sqlalchemy import text
from datetime import datetime, timedelta

s = settings()
cfg = load_pipeline_config()
MIN_TIME = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
print(f'DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}')

DB = medali_dev@localhost:15432/accent_data


## 2. Inputs

In [2]:
with get_engine().connect() as conn:
    end_time = conn.execute(text('SELECT MAX(date_operation) FROM staging.maintenance')).scalar_one()
    counts = {
        'maintenance':  conn.execute(text('SELECT COUNT(*) FROM staging.maintenance')).scalar_one(),
        'offense':      conn.execute(text('SELECT COUNT(*) FROM staging.offense')).scalar_one(),
        'sinistre':     conn.execute(text('SELECT COUNT(*) FROM staging.sinistre')).scalar_one(),
        'reparation':   conn.execute(text('SELECT COUNT(*) FROM staging.reparation')).scalar_one(),
    }
print('staging.maintenance max date_operation =', end_time)
print('row counts:', counts)

staging.maintenance max date_operation = 2026-03-04 23:00:00
row counts: {'maintenance': 1007, 'offense': 4, 'sinistre': 0, 'reparation': 815}


## 3. Execute

In [3]:
import importlib, time, pandas as pd
import accent_fleet.transforms.facts as facts_module
from accent_fleet.pipeline.run_log import begin_run, end_run
facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

run_id = begin_run(mode='notebook:09_load_fact_maintenance')
progress, cursor, t0 = [], MIN_TIME, time.time()
try:
    while cursor < end_time:
        nxt = min(cursor + timedelta(days=CHUNK_DAYS), end_time)
        # Header MUST run before lines for the same window.
        h = load_fact_incremental('fact_maintenance',      run_id=run_id, window_end=nxt)
        l = load_fact_incremental('fact_maintenance_line', run_id=run_id, window_end=nxt)
        progress.append({'window_start': cursor, 'window_end': nxt,
                         'header_rows': h.rows_loaded, 'line_rows': l.rows_loaded})
        cursor = nxt
    end_run(run_id, status='success', rows_loaded=sum(p['header_rows']+p['line_rows'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc)); raise
print(f'done in {time.time()-t0:.1f}s')
pd.DataFrame(progress).tail(10)

2026-04-29 23:38:50 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 31, 0, 0) fact=fact_maintenance run_id=32 start=datetime.datetime(2019, 10, 1, 0, 0)
2026-04-29 23:38:51 [info     ] fact_load.done                 fact=fact_maintenance new_watermark=datetime.datetime(2019, 10, 31, 0, 0) rows=20
2026-04-29 23:38:51 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 31, 0, 0) fact=fact_maintenance_line run_id=32 start=datetime.datetime(2019, 10, 1, 0, 0)
2026-04-29 23:38:52 [info     ] fact_load.done                 fact=fact_maintenance_line new_watermark=datetime.datetime(2019, 10, 31, 0, 0) rows=2
2026-04-29 23:38:52 [info     ] fact_load.start                end=datetime.datetime(2019, 11, 30, 0, 0) fact=fact_maintenance run_id=32 start=datetime.datetime(2019, 10, 30, 23, 50)
2026-04-29 23:38:53 [info     ] fact_load.done                 fact=fact_maintenance new_watermark=datetime.datetime(2019, 11, 30, 0, 0) rows=11
2026-04-29 23

,window_start,window_end,header_rows,line_rows
69,2025-06-01,2025-07-01 00:00:00,0,0
70,2025-07-01,2025-07-31 00:00:00,4,0
71,2025-07-31,2025-08-30 00:00:00,3,0
72,2025-08-30,2025-09-29 00:00:00,3,0
73,2025-09-29,2025-10-29 00:00:00,33,23
74,2025-10-29,2025-11-28 00:00:00,37,29
75,2025-11-28,2025-12-28 00:00:00,34,31
76,2025-12-28,2026-01-27 00:00:00,10,4
77,2026-01-27,2026-02-26 00:00:00,10,0
78,2026-02-26,2026-03-04 23:00:00,2,0


## 4. Inspect

In [4]:
import pandas as pd
with get_engine().connect() as conn:
    h_total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_maintenance')).scalar_one()
    l_total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_maintenance_line')).scalar_one()
    line_mix = pd.read_sql(text('''SELECT line_type, COUNT(*) AS n
                                    FROM warehouse.fact_maintenance_line
                                    GROUP BY 1 ORDER BY 2 DESC'''), conn)
    cost = pd.read_sql(text('''SELECT
        COUNT(*) FILTER (WHERE total_cost > 0) AS rows_with_cost,
        AVG(total_cost) FILTER (WHERE total_cost > 0) AS avg_cost,
        SUM(total_cost) AS total_cost FROM warehouse.fact_maintenance'''), conn)
print(f'fact_maintenance rows: {h_total:,}  fact_maintenance_line rows: {l_total:,}')
display(line_mix); display(cost)

fact_maintenance rows: 820  fact_maintenance_line rows: 707


,line_type,n
0,reparation,704
1,offense,3


,rows_with_cost,avg_cost,total_cost
0,98,671.694245,65826.036
